## 문서 로딩 & 청킹
- pdf를 langchain 로더를 활용해서 문서 형태로 변환
- 정리한 페이지를 csv로 저장해서 -> 재사용도 가능한 형태로 바꿔도 보자!

### 문서를 통째로(268) 한번에 넣을 경우 생기는 문제
1. 임베딩 모델 입력한도
    - small -> 8000 정도 토큰이 최대
    - 나눠서 넣어야 (나누는 기준은 자료 직접 보고 판단! 제목이나 페이지나...)
    - 주로 페이지 읽어서 markdown 형태로 바꾸고 그 기준으로 정리하는 게 합리적
2. 한 문서를 통째로 넣으면 주제가 섞여있을 때 
    - 벡터화 하더라도 주제에 따라서 분류하기 적절하지 않다
    - 한 페이지에 여러 주제가 있다면 -> 벡터화 했을 때 어디로?
    예) 한 페이지에 1. 청년 월세 지원, 2. 노인 복지 지원 주제가 있다면
        -> 분리해서 벡터화하는 게 맞다!

In [5]:
from pypdf import PdfReader

reader = PdfReader("../data/16-1_K희망사다리2026_모두의정책.pdf")
total_k_ladder = len(reader.pages)
total_k_ladder

268

In [4]:
reader.pages[0]

{'/ArtBox': [0.0, 0.0, 430.866, 632.126],
 '/BleedBox': [0.0, 0.0, 430.866, 632.126],
 '/Contents': {'/Filter': '/FlateDecode'},
 '/CropBox': [0.0, 0.0, 430.866, 632.126],
 '/MediaBox': [0.0, 0.0, 430.866, 632.126],
 '/Parent': {'/Count': 4,
  '/Kids': [IndirectObject(1, 0, 2867589162352),
   IndirectObject(27, 0, 2867589162352),
   IndirectObject(29, 0, 2867589162352),
   IndirectObject(26110, 0, 2867589162352)],
  '/Parent': {'/Count': 34,
   '/Kids': [IndirectObject(26109, 0, 2867589162352),
    IndirectObject(26111, 0, 2867589162352),
    IndirectObject(26117, 0, 2867589162352),
    IndirectObject(26123, 0, 2867589162352),
    IndirectObject(26129, 0, 2867589162352),
    IndirectObject(26135, 0, 2867589162352),
    IndirectObject(26141, 0, 2867589162352)],
   '/Parent': {'/Count': 268,
    '/Kids': [IndirectObject(26108, 0, 2867589162352),
     IndirectObject(26147, 0, 2867589162352),
     IndirectObject(26178, 0, 2867589162352),
     IndirectObject(26240, 0, 2867589162352),
     I

In [ ]:
# 내용 살짝만 확인
text = reader.pages[3].extract_text()
print(text)
# 이렇게 하면 화면 안에서 위치는 알 수 없어서, html로 변환해 위치 정보까지 받을 수 있다

모두의 정책 
K-희망사다리 2026차
례
2026년 신규 민생지원 제도
008
009
010
011
012
013
014
015
016
018
유아 단계적 무상교육·보육
참전유공자 등 생계지원금 지급
새도약기금
새도약론
청년미래적금
장기간부 도약적금
범죄피해구조금 확대
범죄피해자 긴급 생활안정비
중소기업 직장인 든든한 한 끼
보호대상아동 민간후원 장학사업
지방우대 패키지
020
021
022
023
024
025
026
028
030
농어촌 기본소득 시범사업
고령자 계속고용 장려금 
비수도권기업 지원 확대
노인 일자리 및 사회활동 지원
국민내일배움카드
지역사랑상품권
창업사업화지원(초기·도약패키지)
중소기업 혁신바우처 지원
팁스(TIPS)
청년 일자리 도약장려금 비수도권 
우대지원
숨은 정부지원금 찾기
032
033
034
035
036
037
038
039
040
042
043
044
045
046
048
049
050
기본형 공익직불제
장애수당
장애인연금
저소득 지역가입자 보험료 지원
독거노인·장애인 응급안전안심서비스
여성청소년 생리용품 지원
청소년복지시설 퇴소청소년 
자립지원수당
장애아동수당
위기청소년 특별지원
고용촉진장려금
임신 사전건강관리 지원사업
농업인 건강·연금보험료 지원
영농도우미 지원
저소득 청소년한부모 아동양육 및 
자립지원
저소득 청소년부모 아동양육비 지원
다문화가족 자녀 기초학습·진로설계·
교육활동비
예술인 국민연금 보험료 지원사업
051 내게 필요한 서비스 
키워드로 바로 찾기
따뜻한 동행 
모두가 행복한 사회


In [10]:
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

load_dotenv()

True

### 로드맵
1. 문서 로드
2. 문서 분할
3. ChromaDB 적재
4. RAG 구축

#### 1. 문서 로드

In [11]:
pdf_docs = []

for i,page in enumerate(reader.pages):
    text = page.extract_text()
    pdf_docs.append(Document(page_content=text, 
                             metadata={"source" : "K희망사다리2026_모두의정책",
                                       "page" : i + 1}
                                       ))

pdf_docs[:10]    

[Document(metadata={'source': 'K희망사다리2026_모두의정책', 'page': 1}, page_content='발간등록번호\n11-1371000-100161-01\nK-희망사다리\n모두의 정책\n2026\n국민생활지원 정보 모음집\n●2026년 신규 민생지원 제도  ●지방우대 패키지 \n●내게 필요한 서비스 키워드로 바로 찾기\n●신청해야 받는 숨은 정부지원금 찾기'),
 Document(metadata={'source': 'K희망사다리2026_모두의정책', 'page': 2}, page_content=''),
 Document(metadata={'source': 'K희망사다리2026_모두의정책', 'page': 3}, page_content='K-희망사다리\n모두의 정책\n2026'),
 Document(metadata={'source': 'K희망사다리2026_모두의정책', 'page': 4}, page_content='모두의 정책 \nK-희망사다리 2026차\n례\n2026년 신규 민생지원 제도\n008\n009\n010\n011\n012\n013\n014\n015\n016\n018\n유아 단계적 무상교육·보육\n참전유공자 등 생계지원금 지급\n새도약기금\n새도약론\n청년미래적금\n장기간부 도약적금\n범죄피해구조금 확대\n범죄피해자 긴급 생활안정비\n중소기업 직장인 든든한 한 끼\n보호대상아동 민간후원 장학사업\n지방우대 패키지\n020\n021\n022\n023\n024\n025\n026\n028\n030\n농어촌 기본소득 시범사업\n고령자 계속고용 장려금 \n비수도권기업 지원 확대\n노인 일자리 및 사회활동 지원\n국민내일배움카드\n지역사랑상품권\n창업사업화지원(초기·도약패키지)\n중소기업 혁신바우처 지원\n팁스(TIPS)\n청년 일자리 도약장려금 비수도권 \n우대지원\n숨은 정부지원금 찾기\n032\n033\n034\n035\n036\n037\n038\n039\n040\n042\n043\n044\n045\n046\n048\n0

#### 2. 문서 청킹(분할)
- RecursiveCharacterTextSplitter
- 엔터가 두번이다 -> 문단이다. 1차적으로 문단에서 먼저 자르고, 엔터 기준으로(줄바꿈)으로 자르고, . 기준으로 자르게 시킬 수도 있음
- 이런 걸 알아서 해 주는게 이 스플리터!

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=400, # 사이즈 기준으로 자동 자름
                                          chunk_overlap = 80)   # 일부러 겹치는 부분 있도록

chunks = splitter.split_documents(pdf_docs)
len(chunks)

606

In [14]:
print(chunks[20].page_content)

새도약기금
 1660-0705
새도약기금
010따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도
지원대상 	 •	 정책발표일(2025년	6월	19일)	기준	①	금융회사별로	개인(개인사업자	포함)이	
②	7년	이상	연체	중인	무담보	채무	계좌의	원금	합계액이	③	5,000만	원	이하
인	경우
핵심내용 	 • 	대상채권	일괄매입	후	즉시	추심을	중단하며,	행정	데이터	등을	활용한	상환
능력	심사를	거쳐	채무자의	여건(상환능력	등)에	따라	소각	또는	채무조정	
지원	
	 	 -	상환능력	없음:	소각(최대	5,000만	원)
	 	 -	상환능력	있음:	강화된	채무조정(최대	80%,	최장	10년	분할상환)
	 •	2025년	10월	1일	새도약기금	출범	후	1년간	대상채권	일괄매입	및	순차적


In [ ]:
# 임베딩 및 저장
DB_PATH = "../data/k_ladder_2026"

# 임베딩 모델 설정
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 벡터스토어에 저장
vectorstore = Chroma.from_documents(
    documents=pdf_docs,  # 지금은 페이지 당 글자 많지 않으니 chunks 대신 그냥 오리지날 넣음. 하지만 글자 많으면 이렇게 하면 안됨
    embedding=embeddings,
    collection_name="k_ladder_2026",
    persist_directory=DB_PATH
)

In [ ]:
# 검색 - 상위 10개

rag_docs = vectorstore.similarity_search("청년 지원 월세는 어떻게 신청하나요?", k=10)
rag_docs

[Document(id='2545971e-5a72-4352-aa61-ddc8887545f4', metadata={'source': 'K희망사다리2026_모두의정책', 'page': 74}, page_content='072생애주기별 국민생활 서비스 - 아동·청소년\n1388\n청소년상담\n학교 밖 청소년 지원\n지원대상 \t •\t 9세\t이상\t24세\t이하\t학교\t밖\t청소년\n\t \t -\t\t초·중학교\t입학\t후\t3개월\t이상\t결석하거나\t취학의무를\t유예한\t청소년\n\t \t -\t\t고등학교에서\t제적·퇴학\t처분을\t받거나\t자퇴한\t청소년\t또는\t미진학자\n핵심내용 \t •\t 학교\t밖\t청소년의\t특성에\t맞는\t학업복귀\t및\t진로체험\t프로그램\t제공\n이용방법 \t •청소년\t1388\t포털(1388.go.kr)\t‘학교\t밖\t청소년\t지원’에서\t확인\n\t •\t방문\t신청:\t전국\t학교밖청소년지원센터\n문의처\t •\t 청소년상담1388\n\t \t -\t\t(전화)\t유선전화는\t☎1388,\t휴대전화는\t지역번호+☎1388\n\t \t -\t\t(온라인)\t문자·카카오톡·페이스북·인스타그램·라인·1388.go.kr\n\t \t \t (웹채팅\t및\t게시판)\n구분  내용\n상담 초기\t상담\t및\t욕구파악,\t심리·진로·가족관계\t등\t문제상담\n교육 취학·재입학\t등\t복교\t지원,\t상급학교\t진학\t지원,\t검정고시\t\n지원\t등\n진로 및 취업 직업체험이나\t진로교육활동\t또는\t직접적인\t경제활동\t참여\n나\t취업을\t지원하는\t활동\t등\n자립 생활,\t문화공간,\t의료지원,\t정서지원,\t경제교육,\t법률교육\t등\n건강검진 상담\t및\t진찰,\t혈액검사,\t구강검진\t등(9세\t이상\t18세\t이하\t\n학교\t밖\t청소년)\n기타 프로그램\t참여자\t급식\t지원\t등'),
 Document(id='3fe0a882-2e9b-4178-a660-0a17364aa2da', metadata={'page'

In [20]:
print(rag_docs[1].page_content)

청년미래적금
 1600-5500
금융위원회
012따뜻한 동행 모두가 행복한 사회 - 2026년 신규 민생지원 제도
지원대상 	 •	 일정	소득	이하	만	19~34세	청년(병역	최대	6년	인정)
	 	 - 		일반형:	개인	소득	6,000만	원	이하	소득자	또는	연	매출	3억	원	이하	소상공인	
중	가구	중위소득	200%	이하
	 	 - 		우대형:	개인소득	3,600만	원	이하	중소기업	재직자	또는	연	매출	1억	원	
이하	소상공인	중	가구	중위소득	150%	이하
   ※  일반형 요건을 충족하는 중소기업 신규 재직자는 우대형 분류
핵심내용 	 • 	만기	3년
	 •	납입액(월	50만	원	한도)에	대한	정부기여금	지원(일반형	6%,	우대형	12%)	
및	이자소득	비과세
  ※  개인소득 6,000~7,500만 원 이하는 이자소득 비과세만 부여
이용방법 	 • 	신청	기간:	2026년	6월	이후(추후	안내	예정)
	 •신청	방법:	비대면	가입	신청(추후	안내	예정)
문의처	 •금융위원회(☎1600-5500) 	및	서민금융진흥원
최대 2,000만 원 
이상
3년	만기
